# DENDRA for reinforcement learning: Atari (discrete actions) and MuJoCo (continuous actions)

This notebook trains the same **recurrent PPO agent** with three different memory "cores" and compares them:

| Core | What gives it memory |
| --- | --- |
| **DENDRA** | Hebbian synapses + spiking dendritic E/I neurons + neuromodulators |
| **LSTM** | gated memory cells (the usual choice for recurrent RL) |
| **Memoryless** | none: a CNN/MLP that sees the last 4 frames (the usual choice for Atari) |

**Default environments**
* `atari:Breakout`: the real Atari 2600 game, 84x84 images, discrete actions.
* `mujoco:HalfCheetah-v5-pomdp`: a running robot with 6 continuous actions. Its **velocities are hidden** (only joint positions are visible), so the agent has to remember past positions to work out how fast it moves.

**How to run on Kaggle**
1. *Settings -> Accelerator*: **GPU T4 x2**. *Settings -> Internet*: **on**.
2. *Run All*, or *Save Version -> Save & Run All* to let it run in the background (up to ~12 hours).
3. Each run stops after `MAX_MINUTES`; its results are saved even if it stops early.

**Contents**
1. Install packages
2. Source code (read it here; each part has a short explanation)
3. Quick check
4. Settings
5. Train all runs
6. Results

## 1. Install packages

In [ ]:
!pip install -q "gymnasium>=1.1" ale-py mujoco minatar

import os
import torch
import gymnasium, ale_py, mujoco
print(f"gymnasium {gymnasium.__version__} | ale-py {ale_py.__version__} | mujoco {mujoco.__version__} | "
      f"torch {torch.__version__} | GPUs: {torch.cuda.device_count()}")

## 2. Source code

Several runs train **at the same time** (one process each, spread over both GPUs), so the code is saved as files and started from the command line.
Each `%%writefile` cell below saves one part of a file; `%%writefile -a` appends to it. Nothing is trained in this section.

| File | Contents |
| --- | --- |
| `dendra/model.py` | DENDRA's biological building blocks (only the parts the RL agent uses; the full file with the language model is in the GitHub repo) |
| `dendra/rl.py` | DENDRA as an RL memory (`DendraCore`) and a ready-made agent (`DendraAgent`) |
| `experiments/rl/ppo.py` | environments, the three cores, the agent and the PPO training loop |
| `experiments/rl/plot_ppo.py` | learning-curve figure and summary table |

In [ ]:
os.makedirs("dendra", exist_ok=True)
os.makedirs("experiments/rl", exist_ok=True)
os.makedirs("runs", exist_ok=True)

In [ ]:
%%writefile dendra/__init__.py
"""DENDRA building blocks (RL notebook version: only model.py is needed)."""

### 2.1 `dendra/model.py`: spikes and divisive normalization

* **`spike`**: a neuron fires (1) when its membrane potential crosses the threshold, otherwise 0. A step function has no useful gradient, so training uses a smooth *surrogate gradient* `1 / (1 + 2|x|)^2` in the backward pass.
* **`DivisiveNorm`**: divides activity by its overall energy, like a pool of inhibitory interneurons (plays the role of LayerNorm/RMSNorm).

In [ ]:
%%writefile dendra/model.py
"""
DENDRA: a language model built from biological neuron mechanisms.

Main components
  LIF neuron + refractory period -> DendriticSpikingLayer (0/1 spikes, soft reset, adaptive threshold)
  Nonlinear dendrites            -> each neuron has B branches, each with its own local nonlinearity
  E (80%) / I (20%) populations  -> Dale's law: E->I and I->E weights are positive; I subtracts from E (soft k-WTA)
  Hebb + STDP + NMDA             -> HebbianSTDP: S <- lam*S + w * v k^T  (replaces attention)
  Neuromodulators (NE, ACh)      -> Neuromodulators: global gain and memory-write gate
  Feedback connections (L6)      -> several up/down passes, higher regions send signals down
  Spinal reflex                  -> early-exit head after region 1
  Homeostasis                    -> loss keeping the firing rate near a target (in the training script)

Three computation modes for the same weights (outputs match, max error < 1e-5):
  forward()      parallel over time, used for training
  prefill()      reads a prompt chunk by chunk, O(n * C)
  step_token()   generates one token with a constant-size state, O(1) memory per token
"""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---------------------------------------------------------------- spikes + surrogate gradient
class _Spike(torch.autograd.Function):
    """All-or-none: spike = 1 when the membrane potential crosses threshold. Surrogate gradient: fast sigmoid."""

    @staticmethod
    def forward(ctx, x, alpha):
        ctx.save_for_backward(x)
        ctx.alpha = alpha
        return (x > 0).to(x.dtype)

    @staticmethod
    def backward(ctx, g):
        (x,) = ctx.saved_tensors
        return g / (1.0 + ctx.alpha * x.abs()) ** 2, None


def spike(x, alpha=2.0):
    return _Spike.apply(x, alpha)


class DivisiveNorm(nn.Module):
    """Divisive normalization by an interneuron pool: divide by the mean activity energy."""

    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.g

### 2.2 Hebbian-STDP synaptic memory (replaces self-attention)

Each head keeps a small synaptic matrix `S` that is written as the sequence goes:

`S_t = lam * S_{t-1} + w_t * v_t k_t^T`  (Hebb: "cells that fire together wire together")

`o_t = S_t q_t`  (reading the memory)

* `lam < 1` makes old memories fade; each head has its own learned time scale (4 to 256 steps).
* `w_t` is the write strength: an NMDA-like gate times acetylcholine from the neuromodulators.
* `forward` (whole sequence), `chunked` and `step` (one step, constant memory) give the same result. The RL agent uses `step` while playing.

In [ ]:
%%writefile -a dendra/model.py


# ---------------------------------------------------------------- L2/3: Hebbian/STDP synapses
class HebbianSTDP(nn.Module):
    """
    Fast-weight memory that replaces attention.
      S_t = lam * S_{t-1} + w_t * v_t k_t^T        (Hebb: pre and post active together -> stronger synapse)
      o_t = S_t q_t
    - lam < 1: synaptic decay; each head has its own time constant (4..256 tokens), learned.
    - Parallel form: o = ((Q K^T) * D * w) V with D_ij = lam^(i-j) for j <= i. This is a causal STDP window
      (pre-before-post strengthens, decaying with distance), so no position embedding is needed.
    - w_t = NMDA(x_t) * ACh_t: write only when the postsynaptic neuron is depolarized (NMDA gate) and ACh is high.
    """

    def __init__(self, d, n_heads, tau_min=4.0, tau_max=256.0):
        super().__init__()
        assert d % n_heads == 0
        self.h, self.dh = n_heads, d // n_heads
        self.qkv = nn.Linear(d, 3 * d, bias=False)
        self.nmda = nn.Linear(d, n_heads)  # coincidence-detector gate
        self.og = nn.Linear(d, d, bias=False)  # output gate (postsynaptic depolarization)
        self.out = nn.Linear(d, d, bias=False)
        self.onorm = DivisiveNorm(self.dh)
        tau = torch.exp(torch.linspace(math.log(tau_min), math.log(tau_max), n_heads))
        lam = 1.0 - 1.0 / tau
        self.lam_logit = nn.Parameter(torch.log(lam / (1 - lam)))  # learnable
        nn.init.constant_(self.nmda.bias, 2.0)

    def lam(self):
        return torch.sigmoid(self.lam_logit)  # [h]

    def _proj(self, x, ach):
        B, T, _ = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, self.h, self.dh).unbind(2)  # [B,T,h,dh]
        q = q * self.dh ** -0.5
        k = F.normalize(k, dim=-1)
        w = torch.sigmoid(self.nmda(x))  # [B,T,h]
        if ach is not None:
            w = w * ach
        return q, k, v, w

    def forward(self, x, ach=None):
        B, T, _ = x.shape
        q, k, v, w = self._proj(x, ach)
        idx = torch.arange(T, device=x.device)
        dist = (idx[:, None] - idx[None, :]).float()  # i - j
        causal = dist >= 0
        loglam = torch.log(self.lam())[:, None, None]  # [h,1,1]
        D = torch.where(causal, torch.exp(loglam * dist.clamp(min=0)), torch.zeros((), device=x.device))
        A = torch.einsum("bihd,bjhd->bhij", q, k) * D  # [B,h,T,T]
        A = A * w.permute(0, 2, 1)[:, :, None, :]  # write strength of each token j
        o = torch.einsum("bhij,bjhd->bihd", A, v)
        return self._finish(x, o)

    def _finish(self, x, o):
        B, T = x.shape[:2]
        o = self.onorm(o).reshape(B, T, -1)
        return self.out(o * F.silu(self.og(x)))

    def chunked(self, x, ach, S, C=128):
        """Read a sequence in chunks of C tokens: parallel inside a chunk, state S carried between chunks.
        Same result as forward() but O(n*C) instead of O(n^2); also returns the final state."""
        B, T, _ = x.shape
        q, k, v, w = self._proj(x, ach)
        lam = self.lam()  # [h]
        loglam = torch.log(lam)
        outs = []
        for s0 in range(0, T, C):
            qc, kc, vc, wc = q[:, s0:s0 + C], k[:, s0:s0 + C], v[:, s0:s0 + C], w[:, s0:s0 + C]
            n = qc.shape[1]
            idx = torch.arange(n, device=x.device)
            dist = (idx[:, None] - idx[None, :]).float()
            D = torch.where(dist >= 0, torch.exp(loglam[:, None, None] * dist.clamp(min=0)), torch.zeros((), device=x.device))
            A = torch.einsum("bihd,bjhd->bhij", qc, kc) * D * wc.permute(0, 2, 1)[:, :, None, :]
            o = torch.einsum("bhij,bjhd->bihd", A, vc)  # within-chunk part
            decay_in = torch.exp(loglam[None, :] * (idx[:, None] + 1).float())  # lam^(a+1) [n,h]
            o = o + torch.einsum("bhde,bihe->bihd", S.to(qc.dtype), qc) * decay_in[None, :, :, None].to(qc.dtype)
            decay_out = torch.exp(loglam[None, :] * (n - 1 - idx)[:, None].float())  # lam^(n-1-b) [n,h]
            wv = vc * (wc * decay_out[None].to(wc.dtype))[..., None]
            S = lam[None, :, None, None] ** n * S + torch.einsum("bjhd,bjhe->bhde", wv.float(), kc.float())
            outs.append(o)
        return self._finish(x, torch.cat(outs, 1)), S

    def step(self, x, ach, S):
        """One-token inference: constant-size memory S [B,h,dh,dh], no KV cache."""
        q, k, v, w = self._proj(x, ach)  # x [B,1,d]
        q, k, v, w = q[:, 0], k[:, 0], v[:, 0], w[:, 0]
        lam = self.lam()[None, :, None, None]
        S = lam * S + w[..., None, None] * torch.einsum("bhd,bhe->bhde", v, k)
        o = torch.einsum("bhde,bhe->bhd", S, q)[:, None]
        return self._finish(x, o), S

### 2.3 Spiking dendritic neurons (layer 5)

* Excitatory (E) neurons each have 4 **dendritic branches** with their own nonlinearity, so a single neuron can compute more than a weighted sum.
* E neurons are leaky integrate-and-fire neurons, run for `T = 4` ticks per step, with a soft reset and an adaptive threshold (a refractory effect).
* Inhibitory (I) neurons follow **Dale's law**: E->I and I->E weights are always positive (softplus), and I neurons only subtract.
* The output is the E firing rate. `last_rate` is kept for the homeostasis loss, which holds firing rates near 15%.

In [ ]:
%%writefile -a dendra/model.py


# ---------------------------------------------------------------- L5: spiking dendritic neurons
class DendriticSpikingLayer(nn.Module):
    """
    E population with dendrites + inhibitory I population, run for T gamma ticks per token.
      branch input    c_b = W_b x                (synapses on each dendritic branch)
      soma input      I = sum_b a_b * g(c_b)     (g: local branch nonlinearity, a_b >= 0)
      E potential     u <- beta*u + gain*I - W_IE s_I
      E spikes        s_E = H(u - (theta + adapt)); soft reset u -= theta*s_E; adapt <- rho*adapt + gamma*s_E
      I population    ui <- beta_i*ui + W_EI s_E;  s_I = H(ui - theta_I)
    Dale's law: W_EI, W_IE >= 0 (softplus). Output = E firing rate -> read out into the residual stream.
    """

    def __init__(self, d, n_e, branches=4, T=4, inh_ratio=0.25, theta=1.0, spike_in=False):
        super().__init__()
        self.spike_in = spike_in
        if spike_in:  # encode the input as ON/OFF spikes (like retinal cells) -> dendritic synapses only add
            self.theta_in_raw = nn.Parameter(torch.full((d,), 1.0))
        self.ne, self.B, self.T = n_e, branches, T
        self.ni = max(1, int(n_e * inh_ratio))  # inhibitory neurons are 20% of the total
        self.dend = nn.Linear(d, n_e * branches)
        self.branch_raw = nn.Parameter(torch.zeros(n_e, branches))  # softplus(0)=0.69
        self.beta_raw = nn.Parameter(torch.full((n_e,), 1.5))  # membrane leak, sigmoid ~ 0.82
        self.w_ei_raw = nn.Parameter(torch.randn(n_e, self.ni) * 0.1 - 4.0)
        self.w_ie_raw = nn.Parameter(torch.randn(self.ni, n_e) * 0.1 - 3.0)
        self.theta, self.theta_i = theta, 1.0
        self.rho, self.gamma = 0.5, 0.5  # refractory period / threshold adaptation
        self.out = nn.Linear(n_e, d, bias=False)
        nn.init.normal_(self.out.weight, std=0.02)

    def encode(self, x):
        th = F.softplus(self.theta_in_raw) + 0.1
        up, un = torch.zeros_like(x), torch.zeros_like(x)
        cp, cn = torch.zeros_like(x), torch.zeros_like(x)
        for _ in range(self.T):
            up = up + x
            un = un - x
            sp, sn = spike(up - th), spike(un - th)
            up, un = up - th * sp, un - th * sn
            cp, cn = cp + sp, cn + sn
        self.last_in_spikes = (cp + cn).sum(-1).mean().detach()  # input spikes per token
        return (cp - cn) / self.T * th  # quantized value, same scale

    def forward(self, x, gain=None):
        Bt = x.shape[:-1]
        if self.spike_in:
            x = self.encode(x)
        c = self.dend(x).view(*Bt, self.ne, self.B)
        I = (F.softplus(self.branch_raw) * F.silu(c)).sum(-1)  # spatial integration over dendrites
        if gain is not None:
            I = I * gain
        beta = torch.sigmoid(self.beta_raw)
        w_ei = F.softplus(self.w_ei_raw)
        w_ie = F.softplus(self.w_ie_raw)
        u = torch.zeros_like(I)
        adapt = torch.zeros_like(I)
        ui = torch.zeros(*Bt, self.ni, device=x.device, dtype=x.dtype)
        s_i = torch.zeros_like(ui)
        count = torch.zeros_like(I)
        count_i = torch.zeros_like(ui)
        for _ in range(self.T):  # T gamma ticks
            u = beta * u + I - s_i @ w_ie
            s = spike(u - (self.theta + adapt))
            u = u - self.theta * s
            adapt = self.rho * adapt + self.gamma * s
            ui = 0.5 * ui + s @ w_ei
            s_i = spike(ui - self.theta_i)
            ui = ui - self.theta_i * s_i
            count = count + s
            count_i = count_i + s_i
        rate = count / self.T
        self.last_rate = rate.mean()
        self.last_rate_i = (count_i / self.T).mean().detach()
        return self.out(rate)

### 2.4 Cortical region and neuromodulators

* **`CorticalRegion`** stacks the parts like a cortical column: L4 (normalization) -> L2/3 (Hebbian memory) -> L5 (spiking neurons), with residual connections.
* **`Neuromodulators`** read a running average of the input and output, for every region, a **gain** (noradrenaline: how excitable the neurons are) and an **ACh** level (acetylcholine: how strongly to write memory).

In [ ]:
%%writefile -a dendra/model.py


# ---------------------------------------------------------------- one cortical region (6-layer column)
class CorticalRegion(nn.Module):
    """L4 input (divisive inhibition) -> L2/3 mixing over time -> L5 spiking output -> L6 receives feedback."""

    def __init__(self, d, n_heads, n_e, branches, T, spike_in=False):
        super().__init__()
        self.l4a = DivisiveNorm(d)
        self.l23 = HebbianSTDP(d, n_heads)
        self.l4b = DivisiveNorm(d)
        self.l5 = DendriticSpikingLayer(d, n_e, branches, T, spike_in=spike_in)
        self.l6_norm = DivisiveNorm(d)
        self.l6 = nn.Linear(d, d, bias=False)  # receives predictions from the region above
        nn.init.zeros_(self.l6.weight)

    def topdown(self, h_above):
        return self.l6(self.l6_norm(h_above))

    def forward(self, h, gain=None, ach=None, fb=None):
        if fb is not None:
            h = h + fb
        h = h + self.l23(self.l4a(h), ach)
        h = h + self.l5(self.l4b(h), gain)
        return h

    def forward_chunked(self, h, gain, ach, fb, S, C=128):
        if fb is not None:
            h = h + fb
        o, S = self.l23.chunked(self.l4a(h), ach, S, C)
        h = h + o
        h = h + self.l5(self.l4b(h), gain)
        return h, S

    def step(self, h, gain, ach, fb, S):
        if fb is not None:
            h = h + fb
        o, S = self.l23.step(self.l4a(h), ach, S)
        h = h + o
        h = h + self.l5(self.l4b(h), gain)
        return h, S


class Neuromodulators(nn.Module):
    """'Brainstem': from a baseline level (causal EMA of the input) emits noradrenaline (gain) and acetylcholine (write gate)."""

    def __init__(self, d, n_regions, n_heads, decay=0.9):
        super().__init__()
        self.L, self.h, self.decay = n_regions, n_heads, decay
        self.norm = DivisiveNorm(d)
        self.proj = nn.Linear(d, n_regions * (1 + n_heads))
        nn.init.zeros_(self.proj.weight)
        nn.init.zeros_(self.proj.bias)  # start: gain = 1, ach = 1

    def _split(self, m):
        m = 2 * torch.sigmoid(self.proj(self.norm(m)))
        m = m.view(*m.shape[:-1], self.L, 1 + self.h)
        return m[..., 0:1], m[..., 1:]  # gain [..,L,1], ach [..,L,h]

    def forward(self, x, K=256):
        # causal EMA as a 1-D convolution (kernel truncated at K steps: decay^K ~ 1e-12), O(n*K) instead of O(n^2)
        B, T, d = x.shape
        K = min(K, T)
        ker = (self.decay ** torch.arange(K - 1, -1, -1, device=x.device, dtype=torch.float32)).to(x.dtype)
        xt = F.pad(x.transpose(1, 2), (K - 1, 0))
        ema = F.conv1d(xt, ker.view(1, 1, K).expand(d, 1, K).contiguous(), groups=d).transpose(1, 2)
        t = torch.arange(T, device=x.device, dtype=torch.float32)
        norm = (1 - self.decay ** (t + 1).clamp(max=K)) / (1 - self.decay)
        return self._split(ema / norm[None, :, None].to(ema.dtype))

    def step(self, x, m_state, n_seen):
        # normalized EMA, identical to the parallel form
        m_state = self.decay * m_state + x
        norm = (1 - self.decay ** (n_seen + 1)) / (1 - self.decay)
        return self._split(m_state / norm), m_state

### 2.5 `dendra/rl.py`: DENDRA as an RL memory

* **`DendraCore`** is the memory: input projection -> 2 cortical regions -> output, plus the neuromodulators.
  * `step()` is used while playing: one step at a time, constant-size memory.
  * `seq()` is used for training: a whole segment of 128 steps **in parallel over time** (about 17x faster), with the same result as 128 calls to `step()`. It also handles episodes that end in the middle of the segment, so memory never leaks from one episode into the next.
  * `homeostasis_loss()` keeps firing rates near 15%.
* **`DendraAgent`** is a ready-made actor-critic for your own projects (discrete or continuous actions). The PPO script below uses only `DendraCore`, inside its own agent that can also swap in an LSTM.

In [ ]:
%%writefile dendra/rl.py
"""
DENDRA for reinforcement learning: a recurrent memory core and a ready-made actor-critic agent,
for discrete actions (Atari, MinAtar, ...) and continuous actions (MuJoCo, Pendulum, ...).

    from dendra.rl import DendraAgent
    agent = DendraAgent(obs_dim=8, n_actions=4)                          # discrete
    agent = DendraAgent(obs_dim=17, n_actions=6, continuous=True)        # continuous
    state = agent.init_state(n_envs)
    dist, value, state = agent.step(obs, state, done)                    # one step for n_envs environments
    action = dist.sample()

The memory of DENDRA is the network state (Hebbian synapses + neuromodulator baseline), not a list of
past observations, so one step costs the same at any episode length. A whole rollout segment can also be
re-run in parallel over time with seq(), which gives the same result as calling step() T times.
"""
import math

import torch
import torch.nn as nn

from .model import CorticalRegion, Neuromodulators


def init_layer(layer, std=math.sqrt(2)):
    """Orthogonal weights and zero bias: the standard initialization for PPO networks."""
    nn.init.orthogonal_(layer.weight, std)
    nn.init.zeros_(layer.bias)
    return layer


class DendraCore(nn.Module):
    """
    DENDRA as an RL memory: a stack of cortical regions (spiking dendritic E/I neurons + Hebbian synapses)
    modulated by neuromodulators. Input: a feature vector per environment; output: a vector of size d.

    Its memory (a dict returned by init_state) has three parts:
      S  [regions, N, heads, dh, dh]  Hebbian synaptic weights of each region (the fast-weight memory)
      m  [N, d]                       running average of the input, read by the neuromodulators
      t  [N]                          steps since the episode began (to normalize that average)
    `done[i] = 1` means environment i starts a new episode at this step: its memory is cleared first.
    """

    def __init__(self, in_dim, d=128, regions=2, heads=4):
        super().__init__()
        self.inp = init_layer(nn.Linear(in_dim, d), 1.0)
        self.regions = nn.ModuleList([CorticalRegion(d, heads, int(1.5 * d), 4, 4) for _ in range(regions)])
        self.nm = Neuromodulators(d, regions, heads)
        self.n_regions, self.n_heads, self.d = regions, heads, d
        self.rates = []                                    # firing rates, collected for the homeostasis loss

    def init_state(self, n, device=None):
        """Empty memory for n environments."""
        device = device or self.inp.weight.device
        dh = self.d // self.n_heads
        return {"S": torch.zeros(self.n_regions, n, self.n_heads, dh, dh, device=device),
                "m": torch.zeros(n, self.d, device=device),
                "t": torch.zeros(n, device=device)}

    def neuromodulator_signals(self, m, t):
        """From the running input average m (after t+1 steps), compute for each region:
        gain (noradrenaline: how excitable the neurons are) and ach (acetylcholine: how strongly to write memory)."""
        decay = self.nm.decay
        normalizer = (1 - decay ** (t + 1)) / (1 - decay)  # sum of the averaging weights so far
        z = 2 * torch.sigmoid(self.nm.proj(self.nm.norm(m / normalizer[..., None])))
        z = z.view(*m.shape[:-1], self.n_regions, 1 + self.n_heads)
        return z[..., :1], z[..., 1:]                      # gain [..., L, 1], ach [..., L, heads]

    def step(self, x, state, done=None):
        """One time step. x [N, in_dim]; done [N] or None. Returns (output [N, d], new state)."""
        if done is None:
            done = torch.zeros(x.shape[0], device=x.device)
        keep = 1 - done
        S = state["S"] * keep[None, :, None, None, None]
        m, t = state["m"] * keep[:, None], state["t"] * keep
        x = self.inp(x)
        m = self.nm.decay * m + x
        gain, ach = self.neuromodulator_signals(m, t)
        h, new_S = x[:, None], []                          # [N, 1, d]: a sequence of length 1
        for l, region in enumerate(self.regions):
            h, S_l = region.step(h, gain[:, None, l], ach[:, None, l], None, S[l])
            new_S.append(S_l)
        self.rates.append(torch.stack([region.l5.last_rate for region in self.regions]))
        return h[:, 0], {"S": torch.stack(new_S), "m": m, "t": t + 1}

    def seq(self, x, state, done=None):
        """
        A whole segment of T steps at once (parallel over time). Same result as calling step() T times,
        but much faster for training. x [T, N, in_dim]; done [T, N] or None. Returns (output [T, N, d], new state).

        The Hebbian memory has a closed form. Inside one episode, the output at step i is
            o_i = sum_{j<=i} lam^(i-j) * w_j * (q_i . k_j) * v_j   +   lam^(i+1) * S0 q_i
        where the second term (the memory carried in from before the segment) only counts
        while no episode boundary has happened yet.
        """
        if done is None:
            done = torch.zeros(x.shape[:2], device=x.device)
        T, N = done.shape
        x = self.inp(x)                                                       # [T, N, d]

        # Neuromodulators: a cheap running average, computed step by step
        m, t, all_m, all_t = state["m"], state["t"], [], []
        for i in range(T):
            keep = 1 - done[i]
            m = self.nm.decay * m * keep[:, None] + x[i]
            t = t * keep
            all_m.append(m); all_t.append(t)
            t = t + 1
        all_m, all_t = torch.stack(all_m), torch.stack(all_t)                 # [T, N, d], [T, N]
        gain, ach = self.neuromodulator_signals(all_m, all_t)                # [T, N, L, .]
        gain, ach = gain.transpose(0, 1), ach.transpose(0, 1)                # [N, T, L, .]

        # Episode bookkeeping
        episode_id = done.transpose(0, 1).cumsum(1)                          # [N, T] resets seen up to step t
        same_episode = episode_id[:, :, None] == episode_id[:, None, :]      # [N, T, T] steps i and j in one episode
        before_first_reset = episode_id == 0                                 # [N, T] still sees the carried-in memory
        in_last_episode = episode_id == episode_id[:, -1:]                   # [N, T] writes into the final memory
        steps = torch.arange(T, device=x.device)
        distance = (steps[:, None] - steps[None, :]).float()                 # [T, T] i - j

        h, new_S, rates = x.transpose(0, 1), [], []                          # h [N, T, d]
        for l, region in enumerate(self.regions):
            memory = region.l23                                               # HebbianSTDP of this region
            x_in = region.l4a(h)
            q, k, v, w = memory._proj(x_in, ach[:, :, l])                    # q,k,v [N,T,heads,dh]; w [N,T,heads]
            log_lam = torch.log(memory.lam())                                 # [heads] per-head decay

            # Within the segment: causal STDP window lam^(i-j), limited to the same episode
            window = torch.where(distance >= 0, torch.exp(log_lam[:, None, None] * distance.clamp(min=0)),
                                 torch.zeros((), device=x.device))            # [heads, T, T]
            window = window[None] * same_episode[:, None]                     # [N, heads, T, T]
            scores = torch.einsum("bihd,bjhd->bhij", q, k) * window * w.permute(0, 2, 1)[:, :, None, :]
            out = torch.einsum("bhij,bjhd->bihd", scores, v)

            # Memory carried in from before the segment, read with decay lam^(i+1)
            S0 = state["S"][l]                                                # [N, heads, dh, dh]
            decay_in = torch.exp(log_lam[None, :] * (steps[:, None] + 1).float())                 # [T, heads]
            out = out + torch.einsum("bhde,bihe->bihd", S0, q) * (decay_in[None, :, :, None] * before_first_reset[:, :, None, None])

            # Memory left at the end of the segment (only steps of the last episode write into it)
            decay_out = torch.exp(log_lam[None, :] * (T - 1 - steps)[:, None].float())            # [T, heads]
            weighted_v = v * (w * decay_out[None] * in_last_episode[:, :, None])[..., None]
            S_end = (torch.exp(log_lam * T)[None, :, None, None] * S0 * (episode_id[:, -1] == 0)[:, None, None, None]
                     + torch.einsum("bjhd,bjhe->bhde", weighted_v, k))

            # The rest of the region: memory output, then the spiking dendritic layer (L5)
            h = h + memory._finish(x_in, out)
            h = h + region.l5(region.l4b(h), gain[:, :, l])
            new_S.append(S_end)
            rates.append(region.l5.last_rate)
        self.rates.append(torch.stack(rates))
        return h.transpose(0, 1), {"S": torch.stack(new_S), "m": all_m[-1], "t": all_t[-1] + 1}

    def clear_rates(self):
        """Forget the firing rates collected so far (call before the forward pass you train on)."""
        self.rates.clear()

    def homeostasis_loss(self, target=0.15):
        """Keeps the firing rate of every region near `target`, like homeostatic plasticity in the brain.
        Uses the rates collected by step()/seq() since the last clear_rates(); add it to the RL loss."""
        rates = torch.stack(self.rates).mean(0)
        return ((rates - target) ** 2).sum()


class DendraAgent(nn.Module):
    """
    Ready-made actor-critic with a DENDRA memory, for discrete or continuous actions:
        observation -> encoder -> DendraCore -> policy distribution + value

    obs_dim     size of a (flat) observation; ignored when you pass your own `encoder`
    n_actions   number of actions (discrete) or action dimensions (continuous)
    continuous  False: Categorical policy; True: Gaussian policy with a learned std
    encoder     optional module for images etc.; it must output vectors of size `encoder_dim`
    """

    def __init__(self, obs_dim, n_actions, continuous=False, d=128, regions=2, heads=4,
                 encoder=None, encoder_dim=None):
        super().__init__()
        self.continuous = continuous
        if encoder is None:                                # small MLP for vector observations
            encoder = nn.Sequential(init_layer(nn.Linear(obs_dim, 64)), nn.Tanh(),
                                    init_layer(nn.Linear(64, 64)), nn.Tanh())
            encoder_dim = 64
        self.encoder = encoder
        self.core = DendraCore(encoder_dim, d, regions, heads)
        self.policy_head = init_layer(nn.Linear(d, n_actions), 0.01)
        self.value_head = init_layer(nn.Linear(d, 1), 1.0)
        if continuous:
            self.log_std = nn.Parameter(torch.zeros(n_actions))

    def init_state(self, n, device=None):
        return self.core.init_state(n, device)

    def distribution(self, h):
        if self.continuous:
            return torch.distributions.Normal(self.policy_head(h), self.log_std.exp())
        return torch.distributions.Categorical(logits=self.policy_head(h))

    def step(self, obs, state, done=None):
        """One step for N environments. obs [N, ...] -> (distribution, value [N], new state)."""
        h, state = self.core.step(self.encoder(obs), state, done)
        return self.distribution(h), self.value_head(h)[:, 0], state

    def seq(self, obs, state, done=None):
        """A whole segment, parallel over time. obs [T, N, ...] -> (distribution, value [T, N], new state)."""
        T, N = obs.shape[:2]
        features = self.encoder(obs.flatten(0, 1)).view(T, N, -1)
        h, state = self.core.seq(features, state, done)
        return self.distribution(h), self.value_head(h)[..., 0], state

    def log_prob(self, dist, action):
        """Log-probability of an action (summed over dimensions for continuous actions)."""
        logp = dist.log_prob(action)
        return logp.sum(-1) if self.continuous else logp

### `experiments/rl/envs.py`: classic-control tasks

Shared task definitions (used here only for the optional classic-control tasks such as `classic:CartPole-v1-pomdp`).

In [ ]:
%%writefile experiments/rl/envs.py
"""
Environments shared by the RL experiments: vectorized, reset inside the step that ends an episode
(so done[i] = 1 means obs[i] is already the first observation of the next episode).

Names
  CartPole-v1, Acrobot-v1, MountainCar-v0, Pendulum-v1     Gymnasium classic control
  ...-pomdp                                                 velocities hidden (positions/angles only)
  minatar:breakout (or another MinAtar game)                10x10 Atari-like game
  atari:Breakout (or another ALE game)                      real Atari 2600, 84x84 grayscale, frame skip 4,
                                                            sticky actions (ALE v5 default), one frame per step
"""
import gymnasium as gym
import numpy as np

# Observation entries kept in the "-pomdp" versions (the velocities are removed)
VISIBLE = {
    "CartPole-v1": [0, 2],            # cart position, pole angle
    "Acrobot-v1": [0, 1, 2, 3],       # cos/sin of both joint angles
    "MountainCar-v0": [0],            # car position
    "Pendulum-v1": [0, 1],            # cos/sin of the angle
}


class KeepEntries(gym.ObservationWrapper):
    def __init__(self, env, keep):
        super().__init__(env)
        self.keep = keep
        low, high = env.observation_space.low[keep], env.observation_space.high[keep]
        self.observation_space = gym.spaces.Box(low, high, (len(keep),), np.float32)

    def observation(self, obs):
        return obs[self.keep].astype(np.float32)


def make_classic(name, frame_stack=1):
    hidden = name.endswith("-pomdp")
    base = name.replace("-pomdp", "")
    env = gym.make(base)
    if hidden:
        env = KeepEntries(env, VISIBLE[base])
    if frame_stack > 1:
        env = gym.wrappers.FlattenObservation(gym.wrappers.FrameStackObservation(env, frame_stack))
    return env


def make_atari(game, frame_stack=1):
    import ale_py
    gym.register_envs(ale_py)
    env = gym.make(f"ALE/{game}-v5", frameskip=1)
    env = gym.wrappers.AtariPreprocessing(env, frame_skip=4, screen_size=84, grayscale_obs=True, noop_max=30)
    if frame_stack > 1:
        env = gym.wrappers.FrameStackObservation(env, frame_stack)
    return env


class VecEnv:
    """n copies of a classic-control or MinAtar environment.
    step(actions) -> obs [n, ...], rewards [n], dones [n] (bool)."""

    def __init__(self, name, n, seed, frame_stack=1):
        self.minatar = name.startswith("minatar:")
        if self.minatar:
            from minatar import Environment
            self.envs = [Environment(name.split(":")[1]) for _ in range(n)]
            for i, env in enumerate(self.envs):
                env.seed(seed * 1000 + i)
                env.reset()
            self.n_actions = self.envs[0].num_actions()
            self.obs_shape = (self.envs[0].state_shape()[2], 10, 10)
            self.continuous = False
            return
        make = (lambda: make_atari(name.split(":")[1], frame_stack)) if name.startswith("atari:") else \
               (lambda: make_classic(name, frame_stack))
        self.vec = gym.vector.SyncVectorEnv([make for _ in range(n)],
                                            autoreset_mode=gym.vector.AutoresetMode.SAME_STEP)
        self.seed = seed
        space = self.vec.single_action_space
        self.continuous = not hasattr(space, "n")
        self.n_actions = space.shape[0] if self.continuous else space.n
        if self.continuous:
            self.low, self.high = space.low, space.high
        self.obs_shape = self.vec.single_observation_space.shape

    def _minatar_obs(self):
        return np.stack([env.state().transpose(2, 0, 1) for env in self.envs]).astype(np.float32)

    def reset(self):
        if self.minatar:
            return self._minatar_obs()
        return self.vec.reset(seed=self.seed)[0].astype(np.float32)

    def step(self, actions):
        if self.minatar:
            rewards, dones = np.zeros(len(self.envs), np.float32), np.zeros(len(self.envs), bool)
            for i, (env, a) in enumerate(zip(self.envs, actions)):
                rewards[i], dones[i] = env.act(int(a))
                if dones[i]:
                    env.reset()
            return self._minatar_obs(), rewards, dones
        if self.continuous:
            actions = np.clip(actions, self.low, self.high)
        obs, rewards, terminated, truncated, _ = self.vec.step(actions)
        return obs.astype(np.float32), rewards.astype(np.float32), terminated | truncated

### 2.6 `ppo.py`: settings

Every setting is a command-line option, so section 5 can start many different runs.
`CONTINUOUS` decides the kind of policy: a Gaussian for continuous actions, a categorical for discrete actions.

In [ ]:
%%writefile experiments/rl/ppo.py
"""
Recurrent PPO with a swappable memory "core", for DISCRETE actions (Atari, MinAtar)
and CONTINUOUS actions (Pendulum, MuJoCo).

The agent is   observation -> Encoder -> Core (memory) -> policy head + value head.
Only the core changes between runs:
  dendra   DENDRA: Hebbian synapses + spiking dendritic E/I neurons + neuromodulators
  lstm     an LSTM cell (the usual choice for recurrent RL)
  mlp      no memory at all (it sees the last 4 frames instead, the usual choice for Atari)

Environments (--env):
  atari:Breakout          real Atari 2600 (ALE), 84x84 grayscale, frame skip 4
  minatar:breakout        miniature 10x10 Atari (MinAtar), fast on CPU
  pendulum-pomdp          Pendulum with the angular velocity HIDDEN, so it must be inferred from memory
  mujoco:HalfCheetah-v5   full MuJoCo; add the suffix "-pomdp" to keep positions only (velocities hidden)
  classic:CartPole-v1     Gymnasium classic control (CartPole-v1, Acrobot-v1, MountainCar-v0, Pendulum-v1),
                          "-pomdp" hides the velocities (see experiments/rl/envs.py)

Example:
  python experiments/rl/ppo.py --env minatar:breakout --core dendra --total 300000 --out runs/ppo.json

File layout
  1. Settings (command line)
  2. Environments (vectorized, reset automatically at the end of an episode)
  3. Networks: encoder, the LSTM and memoryless cores (DENDRA's core is in dendra/rl.py), the agent
  4. PPO: reward normalization, rollout collection, advantages (GAE), policy update
  5. Main training loop and saving of results
"""
import argparse
import json
import math
import os
import sys
import time

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", ".."))
from dendra.rl import DendraCore, init_layer   # the DENDRA memory core (see dendra/rl.py)
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from envs import make_classic                   # classic-control tasks shared with run_brain.py


# ============================================================================ 1. Settings
parser = argparse.ArgumentParser(description="Recurrent PPO with a DENDRA, LSTM or memoryless core")
parser.add_argument("--env", default="minatar:breakout", help="environment, see the list at the top of this file")
parser.add_argument("--core", default="dendra", choices=["dendra", "lstm", "mlp"], help="memory core of the agent")
parser.add_argument("--total", type=int, default=300_000, help="total environment steps")
parser.add_argument("--num_envs", type=int, default=8, help="environments run in parallel")
parser.add_argument("--num_steps", type=int, default=128, help="steps per environment between two updates")
parser.add_argument("--minibatches", type=int, default=4, help="minibatches per epoch (split by environment)")
parser.add_argument("--epochs", type=int, default=None, help="PPO epochs per update; default 4 (discrete), 10 (continuous)")
parser.add_argument("--lr", type=float, default=None, help="learning rate; default 2.5e-4 (discrete), 3e-4 (continuous)")
parser.add_argument("--d", type=int, default=128, help="width of the core")
parser.add_argument("--seed", type=int, default=0)
parser.add_argument("--max_minutes", type=float, default=None, help="stop early after this many minutes (results are still saved)")
parser.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
parser.add_argument("--out", default="runs/ppo.json", help="where to write the JSON results")
args = parser.parse_args()

torch.manual_seed(args.seed)
np.random.seed(args.seed)
torch.set_num_threads(int(os.environ.get("NT", "1")))   # several runs share the CPU, so one thread each by default

DEVICE = torch.device(args.device)
ENV_FAMILY = args.env.split(":")[0]                      # "atari", "minatar", "pendulum-pomdp", "mujoco" or "classic"
CONTINUOUS = ENV_FAMILY in ("pendulum-pomdp", "mujoco") or (ENV_FAMILY == "classic" and "Pendulum" in args.env)

### 2.7 Environments

Several copies of the game run side by side (`--num_envs`), and each copy is restarted as soon as an episode ends.
The `-pomdp` versions **hide velocities**, so the agent has to remember past observations to act well.

In [ ]:
%%writefile -a experiments/rl/ppo.py


# ============================================================================ 2. Environments
class MinAtarVec:
    """Several MinAtar games side by side. Resets a game as soon as it ends. Observations: [N, C, 10, 10]."""

    def __init__(self, game, n, seed):
        from minatar import Environment
        self.envs = [Environment(game) for _ in range(n)]
        for i, env in enumerate(self.envs):
            env.seed(seed * 1000 + i)   # seeds both the sticky-action and the game's random stream (reproducible runs)
            env.reset()
        self.n_actions = self.envs[0].num_actions()
        self.obs_shape = (self.envs[0].state_shape()[2], 10, 10)

    def observe(self):
        return np.stack([env.state().transpose(2, 0, 1) for env in self.envs]).astype(np.float32)

    def reset(self):
        return self.observe()

    def step(self, actions):
        rewards = np.zeros(len(self.envs), np.float32)
        dones = np.zeros(len(self.envs), np.float32)
        for i, (env, action) in enumerate(zip(self.envs, actions)):
            reward, done = env.act(int(action))
            rewards[i], dones[i] = reward, float(done)
            if done:
                env.reset()
        return self.observe(), rewards, dones


class GymVec:
    """Gymnasium environments side by side. A finished episode is reset within the same step,
    so the returned observation is already the first one of the next episode."""

    def __init__(self, make_env_fns, seed):
        self.vec = gym.vector.SyncVectorEnv(make_env_fns, autoreset_mode=gym.vector.AutoresetMode.SAME_STEP)
        self.seed = seed
        space = self.vec.single_action_space
        self.n_actions = space.n if hasattr(space, "n") else space.shape[0]
        self.obs_shape = self.vec.single_observation_space.shape
        if CONTINUOUS:
            self.low, self.high = space.low, space.high

    def reset(self):
        return self.vec.reset(seed=self.seed)[0].astype(np.float32)

    def step(self, actions):
        if CONTINUOUS:
            actions = np.clip(actions, self.low, self.high)
        obs, rewards, terminated, truncated, _ = self.vec.step(actions)
        dones = terminated | truncated
        return obs.astype(np.float32), rewards.astype(np.float32), dones.astype(np.float32)


def keep_first_values(env, k, low, high):
    """Observation wrapper that keeps only the first k numbers (used to hide velocities)."""
    return gym.wrappers.TransformObservation(env, lambda obs: obs[:k].astype(np.float32),
                                             gym.spaces.Box(low, high, (k,), np.float32))


def stack_vector_frames(env, frame_stack):
    """Give a vector observation the last `frame_stack` steps, flattened into one vector."""
    if frame_stack > 1:
        env = gym.wrappers.FrameStackObservation(env, frame_stack)
        env = gym.wrappers.FlattenObservation(env)
    return env


def make_atari(game, frame_stack):
    env = gym.make(f"ALE/{game}-v5", frameskip=1)
    env = gym.wrappers.AtariPreprocessing(env, frame_skip=4, screen_size=84, grayscale_obs=True, noop_max=30)
    return gym.wrappers.FrameStackObservation(env, frame_stack)


def make_pendulum_pomdp(frame_stack):
    env = gym.make("Pendulum-v1")
    env = keep_first_values(env, 2, -1.0, 1.0)             # keep cos(angle), sin(angle); drop angular velocity
    return stack_vector_frames(env, frame_stack)


def make_mujoco(name, frame_stack):
    hide_velocities = name.endswith("-pomdp")
    env = gym.make(name.replace("-pomdp", ""))
    if hide_velocities:
        n_positions = env.unwrapped.model.nq - 1           # joint positions in the observation; velocities follow them
        env = keep_first_values(env, n_positions, -np.inf, np.inf)
    return stack_vector_frames(env, frame_stack)


def make_vec_env(frame_stack):
    n, seed = args.num_envs, args.seed
    if ENV_FAMILY == "minatar":
        return MinAtarVec(args.env.split(":")[1], n, seed)
    if ENV_FAMILY == "atari":
        import ale_py
        gym.register_envs(ale_py)
        game = args.env.split(":")[1]
        return GymVec([lambda: make_atari(game, frame_stack) for _ in range(n)], seed)
    if ENV_FAMILY == "pendulum-pomdp":
        return GymVec([lambda: make_pendulum_pomdp(frame_stack) for _ in range(n)], seed)
    if ENV_FAMILY == "mujoco":
        name = args.env.split(":")[1]
        return GymVec([lambda: make_mujoco(name, frame_stack) for _ in range(n)], seed)
    if ENV_FAMILY == "classic":
        name = args.env.split(":")[1]
        return GymVec([lambda: make_classic(name, frame_stack) for _ in range(n)], seed)
    raise ValueError(f"unknown environment: {args.env}")

### 2.8 Encoder and the two baseline cores

The encoder turns an observation into a feature vector (a CNN for images, a small MLP for vectors).
The LSTM and memoryless cores are the baselines; all cores, DENDRA's included, share one interface: `init_state(n)`, `step(x, state, done)` and, optionally, `seq(...)` for a whole segment.
`done = 1` clears the memory of an environment whose new episode just started.

In [ ]:
%%writefile -a experiments/rl/ppo.py


# ============================================================================ 3. Networks
class Encoder(nn.Module):
    """Turns one observation into a feature vector: a CNN for images, a small MLP for vectors."""

    def __init__(self, obs_shape):
        super().__init__()
        is_image = len(obs_shape) == 3
        if is_image and obs_shape[-1] == 84:              # Atari, 84x84 (the "Nature DQN" CNN)
            self.net = nn.Sequential(
                init_layer(nn.Conv2d(obs_shape[0], 32, 8, 4)), nn.ReLU(),
                init_layer(nn.Conv2d(32, 64, 4, 2)), nn.ReLU(),
                init_layer(nn.Conv2d(64, 64, 3, 1)), nn.ReLU(),
                nn.Flatten(), init_layer(nn.Linear(3136, 512)), nn.ReLU())
            self.dim, self.scale = 512, 1 / 255.0
        elif is_image:                                     # MinAtar, 10x10
            self.net = nn.Sequential(
                init_layer(nn.Conv2d(obs_shape[0], 16, 3, 1)), nn.ReLU(),
                nn.Flatten(), init_layer(nn.Linear(16 * 8 * 8, 128)), nn.ReLU())
            self.dim, self.scale = 128, 1.0
        else:                                              # vector observation
            self.net = nn.Sequential(
                init_layer(nn.Linear(int(np.prod(obs_shape)), 64)), nn.Tanh(),
                init_layer(nn.Linear(64, 64)), nn.Tanh())
            self.dim, self.scale = 64, 1.0

    def forward(self, obs):
        return self.net(obs * self.scale)


# Every core has the same interface:
#   init_state(n)            -> memory for n environments (a dict of tensors)
#   step(x, state, done)     -> (output, new state) for one time step; done[i] = 1 clears environment i's memory first
#   seq(x, state, done)      -> optional: a whole segment of T steps at once, same result as T calls to step()
# The DENDRA core (DendraCore) is part of the package: dendra/rl.py.

class MLPCore(nn.Module):
    """No memory: one dense layer."""

    def __init__(self, in_dim, d):
        super().__init__()
        self.fc = init_layer(nn.Linear(in_dim, d))

    def init_state(self, n):
        return {}

    def step(self, x, state, done):
        out = torch.tanh(self.fc(x)) if CONTINUOUS else F.relu(self.fc(x))
        return out, state


class LSTMCore(nn.Module):
    """Standard LSTM cell."""

    def __init__(self, in_dim, d):
        super().__init__()
        self.cell = nn.LSTMCell(in_dim, d)

    def init_state(self, n):
        zeros = torch.zeros(n, self.cell.hidden_size, device=DEVICE)
        return {"h": zeros, "c": zeros.clone()}

    def step(self, x, state, done):
        keep = (1 - done)[:, None]                         # 0 for environments that start a new episode
        h, c = self.cell(x, (state["h"] * keep, state["c"] * keep))
        return h, {"h": h, "c": c}

### 2.9 The agent (actor-critic)

Encoder -> core -> a policy head (which action to take) and a value head (how good the situation is).

In [ ]:
%%writefile -a experiments/rl/ppo.py


CORES = {"dendra": DendraCore, "lstm": LSTMCore, "mlp": MLPCore}


class Agent(nn.Module):
    """Actor-critic: encoder -> core -> (policy distribution, value)."""

    def __init__(self, obs_shape, n_actions):
        super().__init__()
        self.encoder = Encoder(obs_shape)
        self.core = CORES[args.core](self.encoder.dim, args.d)
        self.policy_head = init_layer(nn.Linear(args.d, n_actions), 0.01)
        self.value_head = init_layer(nn.Linear(args.d, 1), 1.0)
        if CONTINUOUS:
            self.log_std = nn.Parameter(torch.zeros(n_actions))

    def distribution(self, h):
        if CONTINUOUS:
            return torch.distributions.Normal(self.policy_head(h), self.log_std.exp())
        return torch.distributions.Categorical(logits=self.policy_head(h))

    def step(self, obs, state, done):
        """One step for N environments: obs [N, ...]."""
        h, state = self.core.step(self.encoder(obs), state, done)
        return self.distribution(h), self.value_head(h)[:, 0], state

    def seq(self, obs, state, done):
        """A whole segment: obs [T, N, ...]. DENDRA runs in parallel over time; LSTM/MLP loop step by step."""
        T, N = done.shape
        features = self.encoder(obs.flatten(0, 1)).view(T, N, -1)
        if hasattr(self.core, "seq"):
            h, state = self.core.seq(features, state, done)
        else:
            outputs = []
            for t in range(T):
                h_t, state = self.core.step(features[t], state, done[t])
                outputs.append(h_t)
            h = torch.stack(outputs)
        return self.distribution(h), self.value_head(h)[..., 0], state

### 2.10 PPO

Each update does three things:
1. **`collect_rollout`**: play 128 steps in every environment with the current policy.
2. **`compute_advantages`**: estimate how much better each action was than expected (GAE).
3. **`ppo_update`**: improve the policy for several epochs, clipping each change so it stays small.
   Minibatches are groups of whole environments, re-run from the memory they started with (backpropagation through time).

In [ ]:
%%writefile -a experiments/rl/ppo.py


# ============================================================================ 4. PPO
class RewardNormalizer:
    """Divides rewards by the running std of the discounted return (standard for continuous control)."""

    def __init__(self, n_envs, gamma):
        self.returns = np.zeros(n_envs)
        self.gamma = gamma
        self.count, self.mean, self.var = 1e-4, 0.0, 1.0

    def __call__(self, rewards, dones):
        self.returns = self.returns * self.gamma + rewards
        batch_mean, batch_var, batch_count = self.returns.mean(), self.returns.var(), len(rewards)
        delta, total = batch_mean - self.mean, self.count + batch_count
        self.mean += delta * batch_count / total
        self.var = (self.var * self.count + batch_var * batch_count + delta ** 2 * self.count * batch_count / total) / total
        self.count = total
        self.returns[dones > 0] = 0
        return np.clip(rewards / math.sqrt(self.var + 1e-8), -10, 10)


class EpisodeTracker:
    """Adds up rewards per environment and remembers (step, return) of every finished episode."""

    def __init__(self, n_envs):
        self.running = np.zeros(n_envs)
        self.finished = []

    def update(self, rewards, dones, step):
        self.running += rewards
        for i in np.nonzero(dones)[0]:
            self.finished.append((step, self.running[i]))
            self.running[i] = 0

    def mean_of_last(self, n):
        recent = [ret for _, ret in self.finished[-n:]]
        return float(np.mean(recent)) if recent else float("nan")


def clone_state(state):
    return {key: value.clone() for key, value in state.items()}


def collect_rollout(agent, envs, obs, done, state, tracker, reward_fn, global_step):
    """Play num_steps steps in every environment with the current policy and store everything."""
    T, N = args.num_steps, args.num_envs
    action_shape, action_dtype = ((T, N, envs.n_actions), torch.float32) if CONTINUOUS else ((T, N), torch.long)
    buf = {"obs": torch.zeros((T, N) + tuple(envs.obs_shape), device=DEVICE),
           "actions": torch.zeros(action_shape, device=DEVICE, dtype=action_dtype),
           "logp": torch.zeros(T, N, device=DEVICE),
           "rewards": torch.zeros(T, N, device=DEVICE),
           "dones": torch.zeros(T, N, device=DEVICE),        # dones[t] = 1: step t is the first of a new episode
           "values": torch.zeros(T, N, device=DEVICE)}
    for t in range(T):
        buf["obs"][t], buf["dones"][t] = obs, done
        dist, value, state = agent.step(obs, state, done)
        action = dist.sample()
        buf["actions"][t], buf["values"][t] = action, value
        buf["logp"][t] = dist.log_prob(action).sum(-1) if CONTINUOUS else dist.log_prob(action)

        next_obs, reward, next_done = envs.step(action.cpu().numpy())
        tracker.update(reward, next_done, global_step)
        buf["rewards"][t] = torch.tensor(reward_fn(reward, next_done), device=DEVICE)
        obs, done = torch.tensor(next_obs, device=DEVICE), torch.tensor(next_done, device=DEVICE)
        global_step += N
    return buf, obs, done, state, global_step


def compute_advantages(buf, next_value, next_done, gamma, gae_lambda):
    """Generalized Advantage Estimation, from the last step backwards."""
    T = args.num_steps
    advantages = torch.zeros_like(buf["rewards"])
    running = torch.zeros_like(next_value)
    for t in reversed(range(T)):
        if t == T - 1:
            not_done, value_after = 1 - next_done, next_value
        else:
            not_done, value_after = 1 - buf["dones"][t + 1], buf["values"][t + 1]
        td_error = buf["rewards"][t] + gamma * value_after * not_done - buf["values"][t]
        running = td_error + gamma * gae_lambda * not_done * running
        advantages[t] = running
    returns = advantages + buf["values"]
    return advantages, returns


def ppo_update(agent, optimizer, buf, advantages, returns, start_state, clip_coef, ent_coef):
    """Several epochs of clipped PPO. Minibatches are groups of whole environments, so every
    minibatch is re-run as a full sequence from the memory it started with (backprop through time)."""
    N = args.num_envs
    envs_per_batch = N // args.minibatches
    for _ in range(args.epochs):
        order = torch.randperm(N)
        for b in range(args.minibatches):
            idx = order[b * envs_per_batch:(b + 1) * envs_per_batch].to(DEVICE)
            batch_state = {key: (value[:, idx] if key == "S" else value[idx]) for key, value in start_state.items()}
            if args.core == "dendra":
                agent.core.clear_rates()
            dist, values, _ = agent.seq(buf["obs"][:, idx], batch_state, buf["dones"][:, idx])

            logp, entropy = dist.log_prob(buf["actions"][:, idx]), dist.entropy()
            if CONTINUOUS:                                   # independent action dimensions
                logp, entropy = logp.sum(-1), entropy.sum(-1)
            adv = advantages[:, idx]
            adv = (adv - adv.mean()) / (adv.std() + 1e-8)
            ratio = (logp - buf["logp"][:, idx]).exp()
            policy_loss = torch.max(-adv * ratio, -adv * ratio.clamp(1 - clip_coef, 1 + clip_coef)).mean()
            value_loss = 0.5 * ((values - returns[:, idx]) ** 2).mean()
            loss = policy_loss + 0.5 * value_loss - ent_coef * entropy.mean()
            if args.core == "dendra":                        # homeostasis: keep firing rates near 15%
                loss = loss + agent.core.homeostasis_loss(0.15)

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(agent.parameters(), 0.5)
            optimizer.step()

### 2.11 Main loop

Repeats *collect -> advantages -> update*, prints progress about 40 times per run and saves the results to JSON.

In [ ]:
%%writefile -a experiments/rl/ppo.py


# ============================================================================ 5. Main loop
def main():
    N, T = args.num_envs, args.num_steps
    # The memoryless agent gets the last 4 frames instead of a memory (MinAtar frames already show motion)
    hidden_state = ENV_FAMILY in ("atari", "pendulum-pomdp", "mujoco") or args.env.endswith("-pomdp")
    frame_stack = 4 if (args.core == "mlp" and hidden_state) else 1
    envs = make_vec_env(frame_stack)
    agent = Agent(envs.obs_shape, envs.n_actions).to(DEVICE)
    n_params = sum(p.numel() for p in agent.parameters())

    # Hyperparameters: the usual PPO values for Atari (discrete) and MuJoCo (continuous)
    lr = args.lr or (3e-4 if CONTINUOUS else 2.5e-4)
    optimizer = torch.optim.Adam(agent.parameters(), lr=lr, eps=1e-5)
    gamma, gae_lambda = 0.99, 0.95
    clip_coef = 0.2 if CONTINUOUS else 0.1
    ent_coef = 0.0 if CONTINUOUS else 0.01
    args.epochs = args.epochs or (10 if CONTINUOUS else 4)
    n_updates = args.total // (N * T)
    if CONTINUOUS:
        reward_fn = RewardNormalizer(N, gamma)
    elif ENV_FAMILY == "atari":
        reward_fn = lambda reward, done: np.sign(reward)    # reward clipping to -1/0/+1
    else:
        reward_fn = lambda reward, done: reward
    print(f"{args.env} | core {args.core} | {n_params / 1e3:.0f}K params | "
          f"{'continuous' if CONTINUOUS else 'discrete'} ({envs.n_actions}) | {n_updates} updates", flush=True)

    obs = torch.tensor(envs.reset(), device=DEVICE)
    done = torch.zeros(N, device=DEVICE)
    state = agent.core.init_state(N)
    tracker = EpisodeTracker(N)
    curve, start_time, global_step = [], time.time(), 0
    for update in range(1, n_updates + 1):
        if args.max_minutes and time.time() - start_time > args.max_minutes * 60:
            print(f"[{args.core}] time limit of {args.max_minutes} min reached, stopping at step {global_step}", flush=True)
            break
        for group in optimizer.param_groups:                  # learning rate decays linearly to 0
            group["lr"] = lr * (1 - (update - 1) / n_updates)

        start_state = clone_state(state)                       # memory at the start of this segment
        with torch.no_grad():
            buf, obs, done, state, global_step = collect_rollout(agent, envs, obs, done, state, tracker, reward_fn, global_step)
            if args.core == "dendra":
                agent.core.clear_rates()
            _, next_value, _ = agent.step(obs, clone_state(state), done)
            advantages, returns = compute_advantages(buf, next_value, done, gamma, gae_lambda)
        ppo_update(agent, optimizer, buf, advantages, returns, start_state, clip_coef, ent_coef)

        if update % max(1, n_updates // 40) == 0 or update == n_updates:
            elapsed = time.time() - start_time
            avg = tracker.mean_of_last(50)
            curve.append({"step": global_step, "return": avg, "time": elapsed})
            print(f"[{args.core}] step {global_step:8d} | mean return (last 50 episodes) {avg:8.2f} | "
                  f"{global_step / elapsed:5.0f} steps/s | {elapsed / 60:5.1f} min", flush=True)

    last_100 = [ret for _, ret in tracker.finished[-100:]]
    results = {"env": args.env, "core": args.core, "seed": args.seed, "total": global_step, "params": n_params,
               "final_return_mean": float(np.mean(last_100)) if last_100 else None, "curve": curve,
               "episodes": [(int(step), float(ret)) for step, ret in tracker.finished],
               "time_s": time.time() - start_time}
    os.makedirs(os.path.dirname(args.out) or ".", exist_ok=True)
    with open(args.out, "w") as f:
        json.dump(results, f)
    print(f"[{args.core}] DONE: mean return (last 100 episodes) {results['final_return_mean']}, "
          f"{results['time_s'] / 60:.1f} min", flush=True)
    return agent


if __name__ == "__main__":
    agent = main()

### 2.12 `plot_ppo.py`: figure and summary table

In [ ]:
%%writefile experiments/rl/plot_ppo.py
"""
Plot PPO learning curves and print a summary table from the runs/ppo_*.json files
written by ppo.py.

Usage:
  python experiments/rl/plot_ppo.py [results folder, default runs] [output image, default figures/rl_ppo.png]
"""
import glob
import json
import os
import sys
import warnings
from collections import defaultdict

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

RESULTS_DIR = sys.argv[1] if len(sys.argv) > 1 else "runs"
OUTPUT_PATH = sys.argv[2] if len(sys.argv) > 2 else "figures/rl_ppo.png"
warnings.filterwarnings("ignore", "Mean of empty slice")   # early points where no episode has finished yet

# core name -> (legend label, color, line style); the order here is also the drawing order
STYLE = {
    "mlp": ("Memoryless (MLP/CNN)", "#9a9a9a", "--"),
    "lstm": ("LSTM", "#555555", ":"),
    "lstm_big": ("Large LSTM (same #params)", "#222222", "-."),
    "dendra": ("DENDRA", "#2a6fdb", "-"),
}
TITLES = {
    "minatar:breakout": "MinAtar Breakout (discrete)",
    "pendulum-pomdp": "Pendulum, velocity hidden (continuous)",
    "atari:Breakout": "Atari Breakout (discrete)",
    "atari:Pong": "Atari Pong (discrete)",
    "mujoco:HalfCheetah-v5-pomdp": "HalfCheetah, velocities hidden (continuous)",
    "mujoco:HalfCheetah-v5": "HalfCheetah (continuous)",
}


def load_runs(folder):
    """runs[env][core] = list of result dicts (one per seed)."""
    runs = defaultdict(lambda: defaultdict(list))
    for path in glob.glob(os.path.join(folder, "ppo_*.json")):
        with open(path) as f:
            result = json.load(f)
        runs[result["env"]][result["core"]].append(result)
    return runs


def smoothed_curve(result, grid, window=100):
    """Mean return of the last `window` finished episodes, read at every step count in `grid`."""
    episodes = np.array(result["episodes"]) if result["episodes"] else np.zeros((0, 2))
    curve = []
    for step in grid:
        recent = episodes[episodes[:, 0] <= step][-window:, 1]
        curve.append(recent.mean() if len(recent) else np.nan)
    return np.array(curve)


def mean_and_std(values):
    values = np.array([np.nan if v is None else v for v in values], dtype=float)
    if np.all(np.isnan(values)):
        return np.nan, np.nan
    return np.nanmean(values), np.nanstd(values)


def main():
    runs = load_runs(RESULTS_DIR)
    if not runs:
        sys.exit(f"No ppo_*.json files found in {RESULTS_DIR}/")
    envs = sorted(runs)
    fig, axes = plt.subplots(1, len(envs), figsize=(6.5 * len(envs), 4), squeeze=False)
    print(f"{'Environment':38} {'Core':26} {'Params':>8} {'Return, last 100 episodes':>26} {'Steps/s':>9}")

    for ax, env in zip(axes[0], envs):
        total_steps = max(r["total"] for results in runs[env].values() for r in results)
        grid = np.linspace(total_steps / 50, total_steps, 50)
        for core, (label, color, line_style) in STYLE.items():
            results = runs[env].get(core)
            if not results:
                continue
            curves = np.array([smoothed_curve(r, grid) for r in results])
            n_seeds = len(curves)
            seeds_text = f"{n_seeds} seed" + ("s" if n_seeds > 1 else "")
            ax.plot(grid, np.nanmean(curves, 0), line_style, color=color, lw=1.8, label=f"{label} ({seeds_text})")
            if n_seeds > 1:                                   # shaded band: min..max over seeds
                ax.fill_between(grid, np.nanmin(curves, 0), np.nanmax(curves, 0), color=color, alpha=0.12)

            mean, std = mean_and_std([r["final_return_mean"] for r in results])
            steps_per_s = np.mean([r["total"] / r["time_s"] for r in results])
            params = results[0]["params"] / 1e3
            print(f"{env:38} {label:26} {params:7.0f}K {mean:16.2f} ± {std:7.2f} {steps_per_s:9.0f}")

        ax.set_title(TITLES.get(env, env), loc="left", fontsize=11)
        ax.set_xlabel("environment steps")
        ax.set_ylabel("episode return (moving avg. of 100)")
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8, frameon=False)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)

    plt.tight_layout()
    os.makedirs(os.path.dirname(OUTPUT_PATH) or ".", exist_ok=True)
    plt.savefig(OUTPUT_PATH, dpi=150)
    print(f"Figure saved to {OUTPUT_PATH}")


if __name__ == "__main__":
    main()

## 3. Quick check (about 1 minute)

A tiny run of every core on both environments, to catch errors before the long runs start. The results go to `/tmp`, so they do not appear in the final figure.

In [ ]:
import subprocess

for env in ["atari:Breakout", "mujoco:HalfCheetah-v5-pomdp"]:
    for core in ["mlp", "lstm", "dendra"]:
        cmd = ["python", "experiments/rl/ppo.py", "--env", env, "--core", core,
               "--total", "512", "--num_envs", "4", "--num_steps", "32", "--out", "/tmp/check.json"]
        result = subprocess.run(cmd, capture_output=True, text=True)
        status = "OK" if result.returncode == 0 else "ERROR"
        print(f"{status:5} {env:30} {core:6} | {result.stdout.strip().splitlines()[0] if result.stdout else ''}")
        if result.returncode != 0:
            print(result.stderr[-2000:])

## 4. Settings

In [ ]:
JOBS = [                                   # (environment, total environment steps per run)
    ("atari:Breakout", 2_000_000),
    ("mujoco:HalfCheetah-v5-pomdp", 1_000_000),
]
CORES = ["mlp", "lstm", "dendra"]          # memory cores to compare
SEEDS = [0]                                # more seeds = firmer conclusions (RL results are noisy)
MAX_MINUTES = 150                          # time limit for each run
PARALLEL = 3                               # runs at the same time (Kaggle: 4 CPU cores, 2 GPUs)
CHECK_EVERY = 60                           # seconds between progress reports

QUICK_TEST = os.environ.get("DENDRA_QUICK") == "1"   # tiny smoke test of the whole notebook
if QUICK_TEST:
    JOBS = [("atari:Breakout", 2048), ("mujoco:HalfCheetah-v5-pomdp", 2048)]
    MAX_MINUTES, CHECK_EVERY = 5, 5

PLAN = [(env, steps, core, seed) for env, steps in JOBS for seed in SEEDS for core in CORES]
print(f"{len(PLAN)} runs:")
for env, steps, core, seed in PLAN:
    print(f"  {env:30} {core:6} seed {seed}  {steps:,} steps")

## 5. Train all runs

Up to `PARALLEL` runs train at the same time, spread over the GPUs in turn. Every `CHECK_EVERY` seconds this cell prints the latest line of each run's log.
Full logs are in `runs/*.log`; you can stop the cell at any time, and the runs that already finished keep their results.

In [ ]:
import time

N_GPUS = torch.cuda.device_count()


def run_name(env, core, seed):
    return f"runs/ppo_{env.replace(':', '_')}_{core}_s{seed}"


def start_run(env, steps, core, seed, device):
    name = run_name(env, core, seed)
    cmd = ["python", "experiments/rl/ppo.py", "--env", env, "--core", core, "--seed", str(seed),
           "--total", str(steps), "--device", device, "--max_minutes", str(MAX_MINUTES), "--out", name + ".json"]
    log_file = open(name + ".log", "w")
    process = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT, env={**os.environ, "NT": "1"})
    print(f"> start  {env:30} {core:6} seed {seed} on {device}", flush=True)
    return {"process": process, "log_file": log_file, "env": env, "core": core, "seed": seed, "name": name}


def last_log_lines(run, n=1):
    with open(run["name"] + ".log") as f:
        lines = f.read().strip().splitlines()
    return lines[-n:] if lines else [""]


waiting, running, n_started = list(PLAN), [], 0
start_time = time.time()
while waiting or running:
    # Start new runs while there is room
    while waiting and len(running) < PARALLEL:
        env, steps, core, seed = waiting.pop(0)
        device = f"cuda:{n_started % N_GPUS}" if N_GPUS > 0 else "cpu"
        running.append(start_run(env, steps, core, seed, device))
        n_started += 1

    time.sleep(CHECK_EVERY)

    # Report progress and remove finished runs
    still_running = []
    for run in running:
        exit_code = run["process"].poll()
        if exit_code is None:
            still_running.append(run)
            print(f"  {run['env']:30} {run['core']:6} | {last_log_lines(run)[0][:110]}", flush=True)
            continue
        run["log_file"].close()
        if exit_code == 0:
            print(f"done     {run['env']:30} {run['core']:6} seed {run['seed']}: {last_log_lines(run)[0][:110]}", flush=True)
        else:
            print(f"FAILED   {run['env']:30} {run['core']:6} seed {run['seed']} (exit code {exit_code}):", flush=True)
            print("\n".join(last_log_lines(run, 15)), flush=True)
    running = still_running
    print(f"--- {(time.time() - start_time) / 60:.0f} min | {len(running)} running, {len(waiting)} waiting ---", flush=True)
print("All runs finished.")

## 6. Results

The table shows the mean return of the last 100 episodes (higher is better); the figure shows the learning curves.

In [ ]:
!python experiments/rl/plot_ppo.py runs rl_ppo.png
from IPython.display import Image, display
display(Image("rl_ppo.png"))

## Tips

* **Other games:** `("atari:Pong", 2_000_000)`, `("atari:SpaceInvaders", 2_000_000)`.
* **Other robots:** `"mujoco:Hopper-v5-pomdp"`, `"mujoco:Walker2d-v5-pomdp"`; remove `-pomdp` to show the velocities (no memory needed).
* **More seeds** (`SEEDS = [0, 1, 2]`) give firmer conclusions, because RL results vary a lot between seeds.
* **Reference:** standard PPO on Breakout usually reaches a few dozen points after ~2M steps (8M frames) and a few hundred after 10M steps.
* Results and logs are in `runs/`; the figure is `rl_ppo.png`.